# Launch overhead and CUDA Graphs
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week3/Launch_Overhead_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 3, Module 1: The PyTorch frontend**

In eager mode, every PyTorch operation is a separate kernel launch. The CPU sends each kernel and its arguments to the GPU, which costs a few microseconds per launch, and each intermediate result is written to GPU memory and read back by the next kernel. When kernels are short, the GPU spends most of its time waiting for the next launch.

The lecture separates two fixes. **CUDA Graphs** record a sequence of kernels once and replay it with one launch, which removes the launch cost but keeps every kernel and every memory round trip. **Compilation** fuses the operations into fewer kernels, which removes both. In this notebook you measure the launch cost on your GPU, then run a chain of 100 small operations four ways and count the kernels and the bytes each way moves.

**What you will do**
1. Measure the time to launch one tiny kernel.
2. Run a chain of 100 elementwise operations in eager mode, with a CUDA Graph, with `torch.compile`, and with `torch.compile(mode="reduce-overhead")`.
3. Count the kernels each version launches.
4. Repeat with a large tensor, where memory traffic matters more than launches.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The notebook takes about two minutes, most of it compiling.

## 1. Check the environment

In [ ]:
import time
import statistics
import torch
from torch.profiler import profile, ProfilerActivity

print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "This notebook needs a GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. The cost of one launch
The cell launches 2000 additions on a one-element tensor and waits for the GPU only at the end. The GPU work is almost nothing, so the time per operation is the time the CPU needs to issue one launch.

The lecture quotes about 2 to 3 microseconds on A100, H100 and GH200 servers. Colab gives you a small, shared CPU, so expect a larger number here.

In [ ]:
a = torch.zeros(1, device="cuda")
for _ in range(100):
    a.add_(1)          # warm-up
torch.cuda.synchronize()

n_launch = 2000
t0 = time.perf_counter()
for _ in range(n_launch):
    a.add_(1)
torch.cuda.synchronize()
launch_us = (time.perf_counter() - t0) / n_launch * 1e6
print(f"Time per eager operation on a 1-element tensor: {launch_us:.1f} microseconds")

This number includes the Python call and the PyTorch dispatcher as well as the CUDA launch itself. That full per-operation cost is what an eager model pays for every operation.

## 3. A chain of 100 small operations
The function multiplies and adds 50 times: 100 elementwise operations, each reading the previous result. The mathematics is trivial. What matters is the number of operations.

The helper `bench` times a function the right way: warm-up calls first, a synchronize before reading the clock, many repeats, and the median. `count_kernels` runs one call under the profiler and counts the GPU kernels it launched.

In [ ]:
def chain(x):
    for _ in range(50):
        x = x * 1.0001 + 0.0001        # two elementwise operations per step
    return x


def bench(fn, repeats=50):
    for _ in range(5):
        fn()                            # warm-up: compile, autotune, capture
    torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1e6)
    return statistics.median(times)


def count_kernels(fn):
    fn()
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CUDA]) as prof:
        fn()
        torch.cuda.synchronize()
    return sum(e.count for e in prof.key_averages() if e.device_type == torch.autograd.DeviceType.CUDA)

## 4. Four ways to run the chain
| Version | What happens |
|---|---|
| eager | 100 kernel launches, 100 writes and reads of the intermediate |
| CUDA Graph | the 100 kernels are recorded once and replayed with one launch |
| `torch.compile` | Inductor fuses the chain into one Triton kernel |
| `torch.compile(mode="reduce-overhead")` | fusion, and the compiled kernels are replayed as a CUDA Graph |

A CUDA Graph needs fixed input and output memory, so the cell copies new data into a static input tensor and replays the recorded graph.

In [ ]:
def build_versions(x):
    versions = {"eager": lambda: chain(x)}

    # CUDA Graph: capture the eager chain once on a side stream, then replay it.
    static_x = x.clone()
    stream = torch.cuda.Stream()
    stream.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(stream):
        for _ in range(3):
            chain(static_x)             # warm-up on the capture stream
    torch.cuda.current_stream().wait_stream(stream)
    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph):
        static_out = chain(static_x)
    versions["CUDA Graph"] = graph.replay

    compiled = torch.compile(chain)
    versions["torch.compile"] = lambda: compiled(x)
    compiled_ro = torch.compile(chain, mode="reduce-overhead")
    versions["reduce-overhead"] = lambda: compiled_ro(x)

    # All four must agree before we compare their speed.
    ref = chain(x)
    graph.replay()
    assert torch.allclose(static_out, ref)
    assert torch.allclose(compiled(x), ref) and torch.allclose(compiled_ro(x), ref)
    return versions


def compare(n_elements):
    x = torch.randn(n_elements, device="cuda")
    versions = build_versions(x)
    bytes_eager = 100 * 2 * x.numel() * x.element_size()   # each op reads and writes the tensor
    bytes_fused = 2 * x.numel() * x.element_size()         # one read and one write
    print(f"Tensor of {n_elements:,} float32 values ({x.numel() * 4 / 1e6:.3f} MB)")
    print(f"{'version':16} {'kernels':>8} {'time per call':>14} {'HBM traffic':>12}")
    for name, fn in versions.items():
        kernels = count_kernels(fn)
        us = bench(fn)
        traffic = {"eager": bytes_eager, "CUDA Graph": bytes_eager,
                   "torch.compile": bytes_fused, "reduce-overhead": 2 * bytes_fused}[name]
        print(f"{name:16} {kernels:8d} {us:11.1f} us {traffic / 1e6:9.2f} MB")
    torch._dynamo.reset()


compare(1024)

**Reading the output**
- Eager launches 100 kernels. Its time per call is on the order of 100 times the per-operation cost from section 2: the GPU work is tiny, and the CPU cannot issue operations faster.
- The CUDA Graph replays the same 100 kernels with one launch from the CPU, so it is several times faster. Nothing was fused, so the memory traffic is unchanged. The GPU still runs 100 kernels back to back.
- `torch.compile` launches one kernel. Its time is set by the remaining cost of calling the compiled function from Python.
- `reduce-overhead` is slower than plain `torch.compile` here. It launches two kernels: a copy of the input into the CUDA Graph's fixed input buffer, then the fused kernel. With only one kernel to launch, there is little launch cost left to save, and the copy and the graph bookkeeping cost more. CUDA Graphs pay off when a compiled region still launches many kernels, such as a whole model's decode step.

## 5. A large tensor
With 16 million values, each operation now moves 128 MB, so the GPU's work is no longer tiny. Launch cost becomes a small share of the time, and the bytes moved decide the speed.

In [ ]:
compare(16 * 1024 * 1024)

**Reading the output**
- The CUDA Graph is now about as slow as eager. It removed the launch cost, but the 100 round trips to GPU memory remain, and they dominate.
- `torch.compile` is about 100 times faster than eager, because fusion keeps the intermediate in registers and reads and writes the tensor once.
- `reduce-overhead` takes about twice as long as `torch.compile`, because its input copy reads and writes the tensor one more time. The traffic column shows it.
- This is the lecture's conclusion: CUDA Graphs fix launch cost, so launch cost alone does not justify compilation. Fusion and memory planning do.

## Try this
1. Change the chain to 10 operations and run sections 4 and 5 again. When does the CUDA Graph stop helping?
2. Print the kernel Inductor generated with `TORCH_LOGS=output_code`, or run `torch._inductor.utils.run_and_get_code(torch.compile(chain), x)`. Find the single fused loop.
3. Estimate the time the large-tensor eager version should take from the traffic column and the T4's 320 GB/s, and compare with the measurement.

## Summary
- Each eager operation costs a fixed amount of CPU time to issue. With small tensors, the GPU waits for launches.
- CUDA Graphs record a sequence of kernels and replay it with one launch. They remove launch cost but not memory traffic.
- Fusion removes both: fewer kernels and fewer round trips to memory.
- On a large tensor, memory traffic decides the speed, and only fusion helps.

Lecture: Week 3, Module 1, slides 21 to 24 (eager versus compiled, kernel launch path, launch overhead).